# DIAMOND：LIBERO-plus 高斯模糊测试

模型仍为 DIAMOND，服务器 YOUR_CLUSTER_HOST / RTX 5090。LIBERO-plus 只提供图像扰动函数。
选择内核 **DIAMOND Atari (Python 3.12)**，点击 **全部运行**；在隐藏密码框输入 UW 密码。
本 Notebook 仅提交 Breakout 冒烟测试：2 案例 × 4 条件 × ALE/WM = 16 条 rollout。
条件为 normal、sham、高斯模糊 severity 1 和 10（原版 σ=1、10 像素）。H=100 决策步。
模糊作用于预处理后的 64×64 RGB 策略输入，世界模型历史保持干净；不改变游戏动力学。
不会自动提交完整测试。Light Conditions 和 Initial State 尚未接入。


In [ ]:
from pathlib import Path
import csv, getpass, json, shlex, time, uuid
import paramiko
from IPython.display import display, HTML, Image
ROOT = Path.cwd()
assert (ROOT/'diamond_libero_plus_runner.py').exists(), '请从项目目录打开 Notebook'
print(json.loads((ROOT/'libero_plus_provenance.json').read_text()))


In [ ]:
# 一次连接。人手输入密码；不会显示或写入输出。
ssh = paramiko.SSHClient()
known = Path.home() / '.ssh' / 'known_hosts_gix'
assert known.exists(), f'缺少已核验主机指纹文件: {known}'
ssh.load_host_keys(str(known))
ssh.set_missing_host_key_policy(paramiko.RejectPolicy())
secret = getpass.getpass('YOUR_NETID UW password (hidden): ')
try:
    ssh.connect('YOUR_CLUSTER_HOST', username='YOUR_NETID', password=secret,
                allow_agent=False, look_for_keys=False, timeout=15, auth_timeout=45, banner_timeout=20)
finally:
    del secret
ssh.get_transport().set_keepalive(30)
def remote(command, check=True):
    _, stdout, stderr = ssh.exec_command(command, timeout=60)
    out, err = stdout.read().decode(), stderr.read().decode()
    status = stdout.channel.recv_exit_status()
    if check and status:
        raise RuntimeError(f'Remote command failed ({status}): {err or out}')
    return out, err, status
print(remote('hostname; whoami')[0])


In [ ]:
run_id=time.strftime('%Y%m%d-%H%M%S')+'-plus-'+uuid.uuid4().hex[:8]
run_root='/mnt/qnap/YOUR_NETID/diamond-test/atari-robustness/'+run_id
remote('mkdir -p -- '+shlex.quote(run_root))
sftp=ssh.open_sftp()
files=['diamond_atari_runner.py','atari_protocol.py','test_atari_protocol.py',
       'diamond_libero_plus_runner.py','libero_plus_image_functions.py',
       'libero_plus_provenance.json','test_libero_plus_sensor.py']
for name in files: sftp.put(str(ROOT/name),run_root+'/'+name)
local_results=ROOT/'atari-results'/run_id
local_results.mkdir(parents=True)
def record_status(**values):
    (local_results/'status.json').write_text(json.dumps(dict(run_root=run_root,**values),indent=2))
record_status(status='uploaded')
print('新实验目录:',run_root)


In [ ]:
def submit(profile):
    script = f"""#!/bin/bash
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --nodelist=YOUR_SLURM_NODE
#SBATCH --cpus-per-task=2
#SBATCH --mem=8G
#SBATCH --time=04:00:00
#SBATCH --job-name=diamond-plus-{profile}
#SBATCH --output={run_root}/{profile}-%j.out
#SBATCH --error={run_root}/{profile}-%j.err
set -euo pipefail
export HF_HOME=/mnt/qnap/YOUR_NETID/diamond-test/.cache/huggingface
export SDL_VIDEODRIVER=dummy
export OMP_NUM_THREADS=2
export CUBLAS_WORKSPACE_CONFIG=:4096:8
cd /mnt/qnap/YOUR_NETID/diamond-test/diamond
PY="$HOME/.local/share/diamond-test/envs/diamond-py310-cu128/bin/python"
"$PY" -m pip install 'numpy==1.26.0' 'scipy==1.11.4' 'scikit-image==0.22.0'
"$PY" -m pip check
"$PY" -m unittest discover -s {run_root} -p 'test_libero_plus_sensor.py'
"$PY" -m unittest discover -s {run_root} -p 'test_atari_protocol.py'
"$PY" {run_root}/diamond_libero_plus_runner.py --profile {profile} --out {run_root}/{profile}
"""
    path = run_root + '/' + profile + '.slurm'
    with sftp.open(path, 'w') as f: f.write(script)
    out, _, _ = remote('sbatch --parsable ' + shlex.quote(path))
    job = out.strip().split(';')[0]
    assert job.isdigit(), out
    record_status(status='submitted', profile=profile, job=job)
    print(f'{profile}: 已提交 job {job}')
    return job

def wait_for_job(profile, job):
    last = None
    try:
        while True:
            queue, _, _ = remote('squeue -h -u YOUR_NETID -o ' + shlex.quote('%i %T %R'))
            state = '\n'.join(line for line in queue.splitlines() if line.split()[0] == job)
            if state != last:
                print(time.strftime('%H:%M:%S'), state.strip() or '作业已退出队列，核对输出')
                last = state
            record_status(status='monitoring', profile=profile, job=job, queue=state.strip())
            if not state.strip(): break
            time.sleep(10)
    except KeyboardInterrupt:
        print(f'监控已中断，Slurm job {job} 仍可能运行；可再次调用 wait_for_job。')
        raise
    # Accounting is disabled on this cluster; require a completion marker, never infer success from queue disappearance.
    _, _, status = remote('test -f ' + shlex.quote(run_root+'/'+profile+'/PASS'), check=False)
    if status:
        out, err, _ = remote('tail -n 50 ' + shlex.quote(run_root+f'/{profile}-{job}.out') + ' ' + shlex.quote(run_root+f'/{profile}-{job}.err'), check=False)
        record_status(status='failed', profile=profile, job=job)
        raise RuntimeError(out + '\n' + err)
    for name in sftp.listdir(run_root+'/'+profile):
        if name.endswith(('.json','.jsonl','.csv','.gif','.npz')) or name == 'PASS':
            target = local_results/profile
            target.mkdir(exist_ok=True)
            sftp.get(run_root+'/'+profile+'/'+name, str(target/name))
    record_status(status='passed', profile=profile, job=job)
    with (local_results/profile/'summary.csv').open(encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    headings = ['game','domain','condition','level','successes','cases','success_rate','ci95_low','ci95_high','drop_vs_normal']
    display(HTML('<table><tr>'+''.join('<th>'+x+'</th>' for x in headings)+'</tr>' + ''.join('<tr>'+''.join('<td>'+r[x]+'</td>' for x in headings)+'</tr>' for r in rows)+'</table>'))
    print('通过：所有案例完成，normal/sham轨迹哈希一致。')


In [ ]:
smoke_job=submit('smoke')
wait_for_job('smoke',smoke_job)


## 下一步
冒烟通过后可手动新建代码单元运行 `full_job=submit('full'); wait_for_job('full',full_job)`。
完整实验为 3 游戏 × 20 案例 × 12 条件 × 2 环境 = 1440 条 rollout；使用原版全部10档 σ=1…10。
每档是模糊核标准差（像素），不是旧实验[0,1]尺度的加性噪声。
结果存入独立目录；旧实验不覆盖。normal/sham 必须轨迹哈希一致。
输出 GIF 是游戏干净画面，不能当作策略实际看到的模糊画面。
